In [ ]:

!pip install -q transformers miditok symusic

In [ ]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("Enable a GPU runtime before continuing.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:

from google.colab import files

uploaded = files.upload()

Saving train_chunks.pt to train_chunks.pt
Saving val_chunks.pt to val_chunks.pt


In [ ]:

from pathlib import Path

DATA_DIR = Path("/content")

train_path = DATA_DIR / "train_chunks.pt"
val_path = DATA_DIR / "val_chunks.pt"

assert train_path.exists(), "train_chunks.pt is missing"
assert val_path.exists(), "val_chunks.pt is missing"

train_chunks = torch.load(
    train_path, map_location="cpu", weights_only=True
)
val_chunks = torch.load(
    val_path, map_location="cpu", weights_only=True
)

print("Training chunks:", len(train_chunks))
print("Validation chunks:", len(val_chunks))
print("Training sequence length:", len(train_chunks[0]))
print("Validation sequence length:", len(val_chunks[0]))

Training chunks: 6274
Validation chunks: 682
Training sequence length: 2048
Validation sequence length: 2048


In [ ]:
from transformers import AutoModelForCausalLM

MODEL_NAME = "Natooz/Maestro-REMI-bpe20k"
DEVICE = "cuda"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to(DEVICE)

print("Model:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Device:", next(model.parameters()).device)
print("Model context length:", model.config.n_positions)

config.json:   0%|          | 0.00/812 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  196MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Model: Natooz/Maestro-REMI-bpe20k
Parameters: 49,118,208
Device: cuda:0
Model context length: 2048


In [ ]:

from torch.utils.data import Dataset
from transformers import default_data_collator

class PianoChunkDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.chunks[idx], dtype=torch.long)
        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": input_ids.clone(),
        }

train_dataset = PianoChunkDataset(train_chunks)
val_dataset = PianoChunkDataset(val_chunks)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

sample = train_dataset[0]
print("Input shape:", sample["input_ids"].shape)
print("Labels shape:", sample["labels"].shape)
print("Token ID range:", sample["input_ids"].min().item(),
      "to", sample["input_ids"].max().item())


Training examples: 6274
Validation examples: 682
Input shape: torch.Size([2048])
Labels shape: torch.Size([2048])
Token ID range: 5 to 19939


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

from transformers import (
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

OUTPUT_DIR = "/content/drive/MyDrive/piano_music_finetune"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Use FP32 model weights; Trainer will use FP16 autocasting.
model = model.float()
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=0.03,
    lr_scheduler_type="cosine",

    fp16=True,
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=25,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

print("Trainer configured.")
print("Effective batch size:", 8)
print("Epochs:", 3)
print("Checkpoint directory:", OUTPUT_DIR)

Trainer configured.
Effective batch size: 8
Epochs: 3
Checkpoint directory: /content/drive/MyDrive/piano_music_finetune


In [ ]:
train_result = trainer.train()

print("Training complete.")
print("Training metrics:", train_result.metrics)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.564281,3.542152
2,3.335222,3.432935
3,3.297491,3.415473


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training complete.
Training metrics: {'train_runtime': 3179.3514, 'train_samples_per_second': 5.92, 'train_steps_per_second': 0.741, 'total_flos': 8749416450097152.0, 'train_loss': 3.7190733413534307, 'epoch': 3.0}


In [ ]:

FINAL_DIR = Path(
    "/content/drive/MyDrive/piano_music_finetune/best_model"
)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_DIR))
model = trainer.model
model.config.use_cache = True
model.eval()

print("Saved best model to:", FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model to: /content/drive/MyDrive/piano_music_finetune/best_model


In [ ]:
from huggingface_hub import hf_hub_download
from miditok import REMI

tokenizer_path = hf_hub_download(
    repo_id="Natooz/Maestro-REMI-bpe20k",
    filename="tokenizer.json",
)

tokenizer = REMI(params=Path(tokenizer_path))

print("Tokenizer loaded.")
print("Vocabulary size:", len(tokenizer))

tokenizer.json:   0%|          | 0.00/1.02M [00:00<?, ?B/s]

Tokenizer loaded.
Vocabulary size: 20000


/usr/local/lib/python3.13/dist-packages/miditok/classes.py:935: UserWarning: Argument nb_tempos has been renamed num_tempos, you should consider to updateyour code with this new argument name.
  return cls(**input_dict, **kwargs)


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

bos_token_id = model.config.bos_token_id
print("BOS token ID:", bos_token_id)

input_ids = torch.tensor(
    [[bos_token_id]],
    dtype=torch.long,
    device=device,
)

with torch.no_grad():
    generated_ids = model.generate(
        input_ids,
        max_new_tokens=2000,
        do_sample=True,
        temperature=2.5,
        top_k=15,
        top_p=0.95,
        pad_token_id=model.config.eos_token_id,
    )

generated_ids = generated_ids[0].tolist()

print("Generated tokens:", len(generated_ids))
print("First 20 IDs:", generated_ids[:20])

BOS token ID: 2
Generated tokens: 2001
First 20 IDs: [2, 144, 733, 5020, 149, 620, 151, 679, 152, 7449, 582, 155, 293, 8507, 160, 7281, 710, 12040, 132, 9525]


In [ ]:

generated_midi = tokenizer.decode([generated_ids])

output_path = Path(
    "/content/drive/MyDrive/piano_music_finetune/finetuned_piano.mid"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

generated_midi.dump_midi(str(output_path))

print("MIDI saved to:", output_path)

MIDI saved to: /content/drive/MyDrive/piano_music_finetune/finetuned_piano.mid
